In [14]:
from langchain_community.llms import Ollama
llm = Ollama(model="qwen3:8b")

In [15]:
from langchain.agents import Tool
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaAPIWrapper()
wikipedia_tool = Tool(
    name="Wikipedia",
    func=wikipedia.run,
    description="A tool for searching Wikipedia."
)
tools = [wikipedia_tool]

In [16]:
from langchain import hub

# Pull a pre-built prompt from the LangChain Hub
prompt = hub.pull("hwchase17/react")

In [17]:
from langchain.agents import create_react_agent

# Create the agent
agent = create_react_agent(llm, tools, prompt)

In [18]:
from langchain.agents import AgentExecutor

# Create an agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Invoke the agent with a question
response = agent_executor.invoke({"input": "What is the capital of Japan and what is its population?"})

# Print the final answer
print(response["output"])



> Entering new AgentExecutor chain...
<think>
Okay, I need to find the capital of Japan and its population. Let me start by recalling what I know. I think the capital is Tokyo, but I'm not entirely sure. To confirm, I should use the Wikipedia tool. Let me search for Japan's capital.

Action: Wikipedia
Action Input: Japan capital
Page: Capital of Japan
Summary: The capital of Japan is Tokyo. Throughout history, the national capital of Japan has been in locations other than Tokyo. The oldest capital is Nara.

Page: Capital punishment in Japan
Summary: Capital punishment is a legal penalty in Japan. The Penal Code of Japan and several laws list 14 capital crimes. In practice, though, it is applied only for aggravated murder. Executions are carried out by long drop hanging, and take place at one of the seven execution chambers located in major cities across the country. The only crime punishable by a mandatory death sentence is instigation of foreign aggression.
Death sentences are usual

## multiple input function



In [60]:
from typing import List
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# 2. Define the tool function
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

llm = ChatOllama(
    model="qwen3:14b",
    validate_model_on_init=True,
    temperature=0,
).bind_tools([multiply])

result = llm.invoke("What is 5 times 3?")
result.content

'<think>\nOkay, the user is asking "What is 5 times 3?" So I need to multiply 5 and 3. Let me check the available tools. There\'s a function called multiply that takes two integers, a and b. The parameters are required, so I need to provide both. The user\'s question is straightforward, just 5 multiplied by 3. I should call the multiply function with a=5 and b=3. No other functions are available, so that\'s the one to use. Let me make sure the JSON is correctly formatted with the arguments as integers. Yep, that should do it.\n</think>\n\n'

In [41]:
from typing import Optional
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

# 1. Define the input schema using Pydantic
class CalculatorInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

# 2. Define the tool function
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

# 3. Create the StructuredTool
# The 'args_schema' links the function to the Pydantic input schema
calculator_tool = StructuredTool.from_function(
    func=multiply,
    name="Calculator",
    description="Multiplies two integers",
    args_schema=CalculatorInput
)

# Place your structured tool in the list of tools for the agent
tools = [calculator_tool]

In [42]:
from langchain_community.chat_models import ChatOllama
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

# Connect to the local Ollama instance
llm = ChatOllama(model="gpt-oss:20b")

# Pull the ReAct prompt
prompt = hub.pull("hwchase17/react")

# Create the agent with your custom tool
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)

In [43]:
# Invoke the agent with a question that requires multiple inputs for the tool
response = agent_executor.invoke({"input": "What is the product of 121 and 4?"})

# Print the final answer
print(response["output"])



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Question: What is the product of 121 and 4?  
Thought: I need to multiply 121 by 4.  
Action: Calculator  
Action Input: 121, 4  

ValidationError: 2 validation errors for CalculatorInput
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='121, 4', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
b
  Field required [type=missing, input_value={'a': '121, 4'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [46]:
from typing import List

from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


@tool
def validate_user(user_id: int, addresses: List[str]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id (int): the user ID.
        addresses (List[str]): Previous addresses as a list of strings.
    """
    return True


llm = ChatOllama(
    model="gpt-oss:20b",
    validate_model_on_init=True,
    temperature=0,
).bind_tools([validate_user])

result = llm.invoke(
    "Could you validate user 123? They previously lived at "
    "123 Fake St in Boston MA and 234 Pretend Boulevard in "
    "Houston TX."
)

if isinstance(result, AIMessage) and result.tool_calls:
    print(result.tool_calls)

[{'name': 'validate_user', 'args': {'addresses': ['123 Fake St in Boston MA', '234 Pretend Boulevard in Houston TX'], 'user_id': 123}, 'id': '4d3f6949-97f2-470d-9bb1-4d59546d1860', 'type': 'tool_call'}]


In [71]:
### From REF: https://medium.com/pythoneers/power-up-ollama-chatbots-with-tools-113ed8229a7a
from langchain_community.llms import Ollama # to use Ollama llms in langchain
from langchain_core.prompts import ChatPromptTemplate # crafts prompts for our llm
from langchain_community.chat_message_histories import\
StreamlitChatMessageHistory # stores message history
from langchain_core.tools import tool # tools for our llm
from langchain.tools.render import render_text_description # to describe tools as a string 
from langchain_core.output_parsers import JsonOutputParser # ensure JSON input for tools
from operator import itemgetter # to retrieve specific items in our chain.


# Set up the LLM which will power our application.
model = Ollama(model='gpt-oss:20b', temperature=0)

@tool
def add(first: int, second: int) -> int:
    "Add two integers."
    return first + second

@tool
def multiply(first: int, second: int) -> int:
    """Multiply two integers together."""
    return first * second

@tool
def converse(input: str) -> str:
    "Provide a natural language response using the user input."
    return model.invoke(input)

tools = [add, multiply, converse]
rendered_tools = render_text_description(tools)
print(rendered_tools)

add(first: int, second: int) -> int - Add two integers.
multiply(first: int, second: int) -> int - Multiply two integers together.
converse(input: str) -> str - Provide a natural language response using the user input.


In [72]:
system_prompt = f"""You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}
Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.
The value associated with the 'arguments' key should be a dictionary of parameters."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

## Create chain
chain = prompt | model | JsonOutputParser()
chain.invoke({'input': 'What is 3 times 23'})

{'name': 'multiply', 'arguments': {'first': 3, 'second': 23}}

In [73]:
chain.invoke({'input': 'How are you today?'})

{'name': 'converse', 'arguments': {'input': 'How are you today?'}}

In [75]:
# Define a function which returns the chosen tool
# to be run as part of the chain.
def tool_chain(model_output):
    tool_map = {tool.name: tool for tool in tools}
    chosen_tool = tool_map[model_output["name"]]
    return itemgetter("arguments") | chosen_tool

chain = prompt | model | JsonOutputParser() | tool_chain
chain.invoke({'input': 'What is 3 times 23'})

69